# 06 · Baseline ii — Woodpecker-style single-agent self-verification

**Purpose.** One VLM (the same frozen model as the verifier) sees the full image plus the detector's numbered candidate list and decides keep/reject for every item in one call. Validation and decision are fused, evidence is global — the structural opposite of the proposed pipeline.

| | |
|---|---|
| **Proposal objective** | Objective 3 (isolate the benefit of role separation) |
| **Inputs** | detection caches |
| **Outputs** | `results/baselines/woodpecker_*.jsonl`, `results/baselines/woodpecker_*.csv` |
| **Runtime (T4)** | ≈2–6 s/image (generation) → held-out ≈30 min, test ≈60 min |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, DRIVE_PROJECT], check=True)
    PROJECT_ROOT = DRIVE_PROJECT
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab already has torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from dvd.config import load_config, save_selected, results_dir
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache
from dvd import paths

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
from dvd.data.coco import prepare_coco, CocoData, load_splits
prepare_coco(cfg)          # after a runtime restart this re-extracts from the zip cached on Drive (~1-2 min)
coco = CocoData(cfg)
splits = load_splits(cfg)
HELDOUT, TEST = splits["heldout"], splits["test"]
print(f"held-out: {len(HELDOUT)} images | test: {len(TEST)} images")

In [ ]:
from dvd.agents.verifier import build_verifier
from dvd.baselines.woodpecker_style import run_woodpecker_stage, build_prompt
from dvd.experiments import SplitData, available_methods, sweep, operating_point, evaluate_row
H = SplitData(cfg, coco, "heldout", HELDOUT); T = SplitData(cfg, coco, "test", TEST)
verifier = build_verifier(cfg)
iid = TEST[0]; im = coco.image(iid); d = T.det[iid]
print(build_prompt(d["labels"][:5], d["boxes"][:5], *im.size))

In [ ]:
wp_h = run_woodpecker_stage(verifier, H.det, HELDOUT, paths.woodpecker_cache(cfg, "heldout"), cfg, coco.image, "woodpecker held-out")
wp_t = run_woodpecker_stage(verifier, T.det, TEST, paths.woodpecker_cache(cfg, "test"), cfg, coco.image, "woodpecker test")
print("answer parse rate (test):", np.mean([r["parse_ok"] for r in wp_t.values()]))
print("example raw answer:\n", next(r["raw"] for r in wp_t.values() if r["raw"]))

In [ ]:
NAME = "Woodpecker-style single agent"
mh, mt = available_methods(cfg, H)[NAME], available_methods(cfg, T)[NAME]
rows_h = sweep(NAME, mh, H, cfg)
best = operating_point(rows_h)
r_t = evaluate_row(mt, T, best, n_boot=1000); r_t.pop("_masks")
pd.DataFrame(rows_h).to_csv(results_dir(cfg, "baselines") / "woodpecker_sweep_heldout.csv", index=False)
save_json({"selected_on_heldout": best, "test": r_t}, results_dir(cfg, "baselines") / "woodpecker_test.json")
pd.DataFrame([r_t])[["tau", "precision", "recall", "f1", "fp", "fp_per_image"]]